# Silver transformation and SCD Type 2

This notebook reads the four Bronze Delta tables, filters malformed records, deduplicates repeated copies of the same content hash, and reconstructs observed versions into Silver SCD Type 2 tables. It preserves different content hashes for a resource as historical versions. Run only after all four ingestion activities succeed.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

RESOURCE_TYPES = ["Patient", "Encounter", "Observation", "Condition"]
spark.conf.set("spark.sql.session.timeZone", "UTC")
run_timestamp = datetime.now(timezone.utc).isoformat()

summary = []


In [ ]:
for resource_type in RESOURCE_TYPES:
    bronze_table = f"bronze.{resource_type}"
    silver_table = f"silver.{resource_type}"

    if not spark.catalog.tableExists(bronze_table):
        raise RuntimeError(f"Missing {bronze_table}; run ingestion for this resource first.")

    bronze = spark.table(bronze_table)

    # Silver quality gate: exclude missing identifiers, payloads, or hashes.
    clean = bronze.filter(
        F.col("resource_id").isNotNull()
        & F.col("resource_json").isNotNull()
        & F.col("content_hash").isNotNull()
    )

    # Repeated incremental scans may return the same payload; one hash is one observed version.
    duplicate_window = Window.partitionBy(
        "resource_type", "resource_id", "content_hash"
    ).orderBy(
        F.to_timestamp("saved_timestamp").asc_nulls_last(),
        F.to_timestamp("extraction_timestamp").asc_nulls_last()
    )
    versions = (
        clean.withColumn("_duplicate_rank", F.row_number().over(duplicate_window))
             .filter(F.col("_duplicate_rank") == 1)
             .drop("_duplicate_rank")
    )

    # Source meta.lastUpdated is preferred. Extraction/save time is only a fallback.
    versions = (
        versions.withColumn("_source_ts", F.to_timestamp("source_last_updated"))
                .withColumn("_extraction_ts", F.to_timestamp("extraction_timestamp"))
                .withColumn("_saved_ts", F.to_timestamp("saved_timestamp"))
                .withColumn(
                    "_base_ts",
                    F.coalesce(F.col("_source_ts"), F.col("_extraction_ts"), F.col("_saved_ts"))
                )
    )

    # Distinct payload hashes with an identical source timestamp are ordered deterministically.
    # A microsecond tie-breaker keeps SCD2 intervals non-zero in that edge case.
    same_timestamp_window = Window.partitionBy(
        "resource_type", "resource_id", "_base_ts"
    ).orderBy(
        F.col("_extraction_ts").asc_nulls_last(),
        F.col("_saved_ts").asc_nulls_last(),
        F.col("content_hash").asc()
    )
    versions = (
        versions.withColumn("_same_timestamp_rank", F.row_number().over(same_timestamp_window))
                .withColumn(
                    "_version_start",
                    F.expr("timestampadd(MICROSECOND, CAST(_same_timestamp_rank - 1 AS INT), _base_ts)")
                )
    )

    resource_window = Window.partitionBy("resource_type", "resource_id").orderBy(
        F.col("_version_start").asc_nulls_first(),
        F.col("content_hash").asc()
    )
    silver = (
        versions.withColumn("valid_from", F.col("_version_start"))
                .withColumn("valid_to", F.lead("_version_start").over(resource_window))
                .withColumn("is_current", F.col("valid_to").isNull())
                .withColumn("scd2_rebuilt_timestamp", F.lit(run_timestamp).cast("timestamp"))
                .drop("_source_ts", "_extraction_ts", "_saved_ts", "_base_ts",
                      "_same_timestamp_rank", "_version_start")
    )

    # Rebuild from all observed Bronze versions so reruns are deterministic and history is retained.
    (silver.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(silver_table))

    current_checks = (
        spark.table(silver_table).groupBy("resource_id")
        .agg(F.sum(F.col("is_current").cast("int")).alias("current_count"))
        .filter(F.col("current_count") != 1).count()
    )
    if current_checks:
        raise RuntimeError(f"{silver_table} has {current_checks} resource IDs without exactly one current version.")

    summary.append({
        "resource_type": resource_type,
        "bronze_rows": bronze.count(),
        "silver_versions": spark.table(silver_table).count(),
        "current_rows": spark.table(silver_table).filter(F.col("is_current") == True).count(),
        "table": silver_table
    })

print("Silver SCD Type 2 summary:")
for row in summary:
    print(row)
